<a href="https://colab.research.google.com/github/brindatenn/phd-prep-summer-2026/blob/main/week1/pytorch_fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyTorch Fundamentals (Tensor Cheat Sheet)

**Tensors** are the fundamental building block of ML/deep learning.
Their job is to represent data numerically.
A model learns by starting with random tensors and repeatedly adjusting them:

`random numbers → tensor operations → adjust → repeat`

In [31]:
import torch
torch.__version__

'2.11.0+cpu'

## 1. Creating tensors

`scalar → vector → matrix → tensor` are all just tensors of different dimensions.

| Name | What it is | `ndim` | Convention |
|------|-----------|--------|------------|
| scalar | single number | 0 | lowercase `a` |
| vector | 1D array of numbers | 1 | lowercase `y` |
| matrix | 2D array of numbers | 2 | UPPERCASE `Q` |
| tensor | n-D array of numbers | any | UPPERCASE `X` |

**Two tricks to remember:**
- `ndim` = number of square brackets `[` on **one** outer side.
- `shape` is read **outside-in** (outermost grouping first). Multiplying the shape values = total element count.

In [32]:
# Scalar - 0 dimensions
scalar = torch.tensor(7)
print(scalar, "| ndim:", scalar.ndim)
scalar.item()          # .item() pulls out the Python number (ONLY works on 1-element tensors)

tensor(7) | ndim: 0


7

In [33]:
# Vector - 1 dimension (one outer bracket)
vector = torch.tensor([7, 7])
print(vector, "| ndim:", vector.ndim, "| shape:", vector.shape)

tensor([7, 7]) | ndim: 1 | shape: torch.Size([2])


In [34]:
# Matrix - 2 dimensions
MATRIX = torch.tensor([[7, 8],
                       [9, 10]])
print(MATRIX, "\n| ndim:", MATRIX.ndim, "| shape:", MATRIX.shape)   # shape [2, 2]

tensor([[ 7,  8],
        [ 9, 10]]) 
| ndim: 2 | shape: torch.Size([2, 2])


In [35]:
# Tensor - 3 dimensions (count the outer brackets: 3)
TENSOR = torch.tensor([[[1, 2, 3],
                        [3, 6, 9],
                        [2, 4, 5]]])
print(TENSOR, "\n| ndim:", TENSOR.ndim, "| shape:", TENSOR.shape)   # shape [1, 3, 3] -> read outside-in

tensor([[[1, 2, 3],
         [3, 6, 9],
         [2, 4, 5]]]) 
| ndim: 3 | shape: torch.Size([1, 3, 3])


## 2. Random tensors, zeros & ones

You rarely build tensors by hand - models **start from random tensors** and adjust them.

`size=` controls the shape.

In [36]:
# Random tensor - default dtype is float32
random_tensor = torch.rand(size=(3, 4))
random_tensor, random_tensor.dtype

(tensor([[0.8694, 0.5677, 0.7411, 0.4294],
         [0.8854, 0.5739, 0.2666, 0.6274],
         [0.2696, 0.4414, 0.2969, 0.8317]]),
 torch.float32)

In [37]:
# Common image shape: [height, width, colour_channels]
image = torch.rand(size=(224, 224, 3))
image.shape, image.ndim

(torch.Size([224, 224, 3]), 3)

In [38]:
zeros = torch.zeros(size=(3, 4))   # often used for masking
ones  = torch.ones(size=(3, 4))
zeros, ones

(tensor([[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]]),
 tensor([[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]]))

## 3. Ranges & "like" tensors

`torch.arange(start, end, step)` → range of values (end is **exclusive**).

`torch.zeros_like(input)` / `torch.ones_like(input)` → same **shape** as another tensor.

In [39]:
zero_to_ten = torch.arange(start=0, end=10, step=1)   # note: torch.range() is deprecated
ten_zeros   = torch.zeros_like(input=zero_to_ten)     # matches shape of zero_to_ten
zero_to_ten, ten_zeros

(tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0]))

## 4. Datatypes, and the 3 attributes that cause most errors

Default dtype is **`float32`**. Lower precision (16/8-bit) = faster but less accurate.

> The 3 things that cause most PyTorch errors:
> - **`shape`** — do the shapes line up?
> - **`dtype`** — are the datatypes the same?
> - **`device`** — are both tensors on the same device (CPU/GPU)?

In [40]:
float_32_tensor = torch.tensor([3.0, 6.0, 9.0],
                               dtype=None,          # None -> defaults to float32
                               device=None,         # None -> default device (CPU)
                               requires_grad=False) # True -> track gradients for this tensor
float_32_tensor.shape, float_32_tensor.dtype, float_32_tensor.device

(torch.Size([3]), torch.float32, device(type='cpu'))

In [41]:
# Inspect any tensor's key attributes
some_tensor = torch.rand(3, 4)
print(f"Shape:  {some_tensor.shape}")
print(f"Dtype:  {some_tensor.dtype}")
print(f"Device: {some_tensor.device}")

Shape:  torch.Size([3, 4])
Dtype:  torch.float32
Device: cpu


## 5. Basic operations

`+  -  *  /` all work element-wise. **Values don't change unless reassigned.**

In [42]:
tensor = torch.tensor([1, 2, 3])
print(tensor + 10)          # add
print(tensor * 10)          # multiply (element-wise)
print(tensor)               # unchanged — original only updates if reassigned

# Element-wise multiply: each element times its match [1*1, 2*2, 3*3]
print(tensor * tensor)      # -> tensor([1, 4, 9])

tensor([11, 12, 13])
tensor([10, 20, 30])
tensor([1, 2, 3])
tensor([1, 4, 9])


## 6. Matrix multiplication (the core operation of neural networks)

Use `torch.matmul()` (aka `torch.mm`, or the `@` operator). Two rules:

1. **Inner dimensions must match:** `(2, 3) @ (3, 2)` **✓**  ·  `(3, 2) @ (3, 2)` **✗**
2. **Result takes the outer dimensions:** `(2, 3) @ (3, 2) → (2, 2)`

Element-wise `*` multiplies matching positions. Matmul **also sums** the products (the dot product).

In [43]:
tensor = torch.tensor([1, 2, 3])
print("Element-wise:", tensor * tensor)            # [1, 4, 9]
print("Matmul (dot):", torch.matmul(tensor, tensor))  # 1*1 + 2*2 + 3*3 = 14
# torch.matmul is far faster than looping by hand — avoid manual for-loops.

Element-wise: tensor([1, 4, 9])
Matmul (dot): tensor(14)


## 7. Shape errors → fix with transpose

Shape mismatch is *the* most common deep learning error.
**Transpose** (`tensor.T`) switches the dimensions so the inner dimensions line up.

In [44]:
tensor_A = torch.tensor([[1, 2], [3, 4], [5, 6]], dtype=torch.float32)  # shape (3, 2)
tensor_B = torch.tensor([[7, 10], [8, 11], [9, 12]], dtype=torch.float32) # shape (3, 2)

# torch.matmul(tensor_A, tensor_B)  # ✗ errors: (3,2) @ (3,2), inner dims 2 != 3

# Transpose B -> (2, 3) so inner dims match: (3,2) @ (2,3) -> shape (3,3)
print("Tensor A:"), print(tensor_A)
print("Tensor B:"), print(tensor_B)
print("Tensor B.T: "), print(tensor_B.T)
print("\n")
output = torch.matmul(tensor_A, tensor_B.T)
print("A:", tensor_A.shape, "B.T:", tensor_B.T.shape, "-> output:", output.shape)
print("Output:")
output

Tensor A:
tensor([[1., 2.],
        [3., 4.],
        [5., 6.]])
Tensor B:
tensor([[ 7., 10.],
        [ 8., 11.],
        [ 9., 12.]])
Tensor B.T: 
tensor([[ 7.,  8.,  9.],
        [10., 11., 12.]])


A: torch.Size([3, 2]) B.T: torch.Size([2, 3]) -> output: torch.Size([3, 3])
Output:


tensor([[ 27.,  30.,  33.],
        [ 61.,  68.,  75.],
        [ 95., 106., 117.]])

## 8. Aggregation — min, max, mean, sum

Go from many values to fewer. **`mean()` needs a float dtype.**
`argmax()` / `argmin()` return the **index** of the max/min (used later with softmax).

In [45]:
x = torch.arange(0, 100, 10)
print("Min:", x.min(), "| Max:", x.max())
print("Mean:", x.type(torch.float32).mean())   # must cast to float, otherwise errors
print("Sum:", x.sum())
print("Index of max:", x.argmax(), "| Index of min:", x.argmin())

Min: tensor(0) | Max: tensor(90)
Mean: tensor(45.)
Sum: tensor(450)
Index of max: tensor(9) | Index of min: tensor(0)


## 9. Change datatype

`tensor.type(dtype)` converts between datatypes to fix dtype mismatches.

In [46]:
tensor = torch.arange(10., 100., 10.)      # float32 by default
tensor_float16 = tensor.type(torch.float16)
print(tensor.dtype, "->", tensor_float16.dtype)

torch.float32 -> torch.float16


## 10. Reshaping, stacking, squeezing, unsqueezing, permuting

Change a tensor's shape/dimensions **without changing its values** — needed to satisfy matmul shape rules.

| Method | Does |
|--------|------|
| `reshape(shape)` | reshape to a compatible shape |
| `view(shape)` | new view that **shares data** with the original (change one → change both) |
| `torch.stack([...], dim)` | stack tensors along a new dimension |
| `squeeze()` | remove all dimensions of size 1 |
| `unsqueeze(dim)` | add a dimension of size 1 at `dim` |
| `permute(dims)` | reorder axes (returns a **view**) |

In [47]:
x = torch.arange(1., 8.)                 # shape [7]
x_reshaped = x.reshape(1, 7)             # add a dimension -> [1, 7]
print("reshaped:", x_reshaped.shape)

z = x.view(1, 7)                         # view SHARES data with x
z[:, 0] = 5                              # this also changes x!
print("view changes original:", x)

reshaped: torch.Size([1, 7])
view changes original: tensor([5., 2., 3., 4., 5., 6., 7.])


In [48]:
x = torch.arange(1., 8.)
x_stacked = torch.stack([x, x, x, x], dim=0)   # stack 4 copies -> shape [4, 7]
print("stacked shape:", x_stacked.shape)

x_reshaped = x.reshape(1, 7)
print("squeeze:  ", x_reshaped.squeeze().shape)          # [1, 7] -> [7]
print("unsqueeze:", x.unsqueeze(dim=0).shape)            # [7] -> [1, 7]

stacked shape: torch.Size([4, 7])
squeeze:   torch.Size([7])
unsqueeze: torch.Size([1, 7])


In [49]:
# permute — great for images: [H, W, C] -> [C, H, W]
x_image = torch.rand(size=(224, 224, 3))
x_permuted = x_image.permute(2, 0, 1)   # axis order 0->1, 1->2, 2->0
print(x_image.shape, "->", x_permuted.shape)

torch.Size([224, 224, 3]) -> torch.Size([3, 224, 224])


## 11. Indexing

Works like NumPy / Python lists, just with more dimensions.
Indexing goes **outer → inner**. Use `:` for "all values in this dimension".

In [50]:
x = torch.arange(1, 10).reshape(1, 3, 3)
print(x)
print("x[0]:      ", x[0])          # first (outer) bracket -> the 3x3 matrix
print("x[0][0]:   ", x[0][0])       # first row
print("x[0][0][0]:", x[0][0][0])    # single element -> 1
print("x[:, 0]:   ", x[:, 0])       # all of dim0, index 0 of dim1
print("x[:, :, 1]:", x[:, :, 1])    # all of dim0 & dim1, index 1 of dim2 -> [2, 5, 8]

tensor([[[1, 2, 3],
         [4, 5, 6],
         [7, 8, 9]]])
x[0]:       tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]])
x[0][0]:    tensor([1, 2, 3])
x[0][0][0]: tensor(1)
x[:, 0]:    tensor([[1, 2, 3]])
x[:, :, 1]: tensor([[2, 5, 8]])


## 12. PyTorch ↔ NumPy

- `torch.from_numpy(ndarray)` → NumPy array to tensor
- `tensor.numpy()` → tensor to NumPy array

⚠️ NumPy defaults to **float64**, PyTorch defaults to **float32**. Cast if needed:
`torch.from_numpy(array).type(torch.float32)`

In [51]:
import numpy as np
array = np.arange(1.0, 8.0)
tensor = torch.from_numpy(array)          # keeps float64 from NumPy
numpy_again = torch.ones(7).numpy()       # tensor -> array (float32)
array.dtype, tensor.dtype, numpy_again.dtype

(dtype('float64'), torch.float64, dtype('float32'))

## 13. Reproducibility — `torch.manual_seed()`

Randomness on a computer is *pseudo*-random. A **seed** "flavours" the randomness so you get the
same values every run → reproducible experiments.

> The seed must be set **again before each** `rand()` call you want to match.

In [52]:
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)
random_C = torch.rand(3, 4)

torch.manual_seed(RANDOM_SEED)   # reset before the next rand() or it won't match
random_D = torch.rand(3, 4)

print(random_C == random_D)      # all True

tensor([[True, True, True, True],
        [True, True, True, True],
        [True, True, True, True]])


## 14. Running tensors on the GPU

GPUs are much faster for the matrix maths neural networks need. Write **device-agnostic code**
so it runs on GPU if available, else CPU.

Key rules:
- `.to(device)` returns a **copy** — reassign it: `tensor = tensor.to(device)`.
- A GPU tensor can't go straight to NumPy — bring it back first: `tensor.cpu().numpy()`.

In Colab: **Runtime → Change runtime type → GPU** to enable one.

In [53]:
# !nvidia-smi   # (uncomment in Colab) check for an available NVIDIA GPU

# Device-agnostic setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
print("GPU count:", torch.cuda.device_count())

Using device: cpu
GPU count: 0


In [54]:
tensor = torch.tensor([1, 2, 3])          # defaults to CPU
tensor_on_gpu = tensor.to(device)         # move to GPU (if available) — returns a copy
print(tensor_on_gpu, tensor_on_gpu.device)

# Back to CPU (needed before converting to NumPy)
tensor_back_on_cpu = tensor_on_gpu.cpu().numpy()
print(tensor_back_on_cpu)

tensor([1, 2, 3]) cpu
[1 2 3]


## Quick recap

- Tensors represent data numerically; scalar/vector/matrix are just tensors of different `ndim`.
- `ndim` = outer brackets; `shape` reads outside-in.
- Default dtype `float32`; most errors come down to **shape / dtype / device**.
- Matmul: inner dims must match, result takes the outer dims — fix mismatches with `.T`.
- `reshape / view / stack / squeeze / unsqueeze / permute` change shape without changing values.
- Use `manual_seed()` for reproducibility and device-agnostic code for GPU/CPU portability.